Hour 1 — Data Modeling Fundamentals

In [3]:
import pandas as pd
import numpy as np
import duckdb

Let’s start by creating a small customer dataset in Pandas.

In [5]:
customers_df = pd.DataFrame({
    "customer_id": [1,2,3],
    "customer_name": ["Alex", "Maria", "James"],
    "state": ["CA","NV","CA"]
})

Products Dimension

In [6]:
products_df = pd.DataFrame({
    "product_id": [101,102,103],
    "product_name": ["Laptop","Monitor","Keyboard"],
    "category": ["Computers","Accessories","Accessories"]
})

Build the Fact Table

In [7]:
orders_df = pd.DataFrame({
    "order_id": [1001,1002,1003,1004,1005],
    "customer_id": [1,2,3,1,2],
    "product_id": [101,102,103,102,101],
    "quantity": [1,2,3,1,1],
    "price": [1200,300,100,300,1200]
})

Hour 2 — Load the Data Model into DuckDB

create a new database so we can build today's model separately:

In [8]:
con = duckdb.connect("../data/day_08_warehouse.duckdb")

Create the Customers Table

In [9]:
con.execute("""
    CREATE OR REPLACE TABLE customers AS
    SELECT
        *
    FROM customers_df
""")

In [11]:
con.execute("""
    CREATE OR REPLACE TABLE products AS
    SELECT 
        *
    FROM products_df
""")

In [12]:
con.execute("""
    CREATE OR REPLACE TABLE orders AS
    SELECT
        *
    FROM orders_df
""")

Verify the Tables

In [13]:
con.execute("""
    SELECT 
        table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

,table_name
0,customers
1,orders
2,products


Join Orders to Customers

In [14]:
orders_view = con.execute("""
    SELECT *
    FROM orders
""").df()

customers_view = con.execute("""
    SELECT *
    FROM customers
""").df()

products_view = con.execute("""
    CREATE OR REPLACE TABLE products AS
    SELECT 
        *
    FROM products_df
""")

display(orders_view)
display(customers_view)


,order_id,customer_id,product_id,quantity,price
0,1001,1,101,1,1200
1,1002,2,102,2,300
2,1003,3,103,3,100
3,1004,1,102,1,300
4,1005,2,101,1,1200


,customer_id,customer_name,state
0,1,Alex,CA
1,2,Maria,NV
2,3,James,CA


In [ ]:
con.execute("""
    SELECT
        order_id,
        customer_name,
        quantity,
        price
    FROM orders AS o
    INNER JOIN customers AS c
        ON o.customer_id = c.customer_id        
""").df()

,order_id,customer_name,quantity,price
0,1001,Alex,1,1200
1,1002,Maria,2,300
2,1003,James,3,100
3,1004,Alex,1,300
4,1005,Maria,1,1200


Add Products

In [17]:
con.execute("""
    SELECT
        order_id,
        customer_name,
        product_name,
        category,
        quantity,
        price
    FROM orders AS o
    INNER JOIN customers AS c
        ON o.customer_id = c.customer_id
    INNER JOIN products AS p
        ON o.product_id = p.product_id        
""").df()

,order_id,customer_name,product_name,category,quantity,price
0,1001,Alex,Laptop,Computers,1,1200
1,1002,Maria,Monitor,Accessories,2,300
2,1003,James,Keyboard,Accessories,3,100
3,1004,Alex,Monitor,Accessories,1,300
4,1005,Maria,Laptop,Computers,1,1200


Hour 3 — Build an Analytics Table from the Star Schema

In [18]:
con.execute("""
    CREATE SCHEMA IF NOT EXISTS analytics
""")

Build order_details

In [19]:
con.execute("""
    CREATE OR REPLACE TABLE analytics.order_details AS
    SELECT
        o.order_id,
        c.customer_name,
        c.state,
        p.product_name,
        p.category,
        o.quantity,
        o.price,
        o.price * o.quantity AS total_sales
    FROM orders AS o
    INNER JOIN customers AS c
        ON o.customer_id = c.customer_id
    INNER JOIN products AS p
        on o.product_id = p.product_id
""")

Verify order_details

Now query the table you just created.

In [20]:
con.execute("""
    SELECT
        order_id,
        customer_name,
        product_name,
        category,
        total_sales
    FROM analytics.order_details
    ORDER BY total_sales DESC
""").df()

,order_id,customer_name,product_name,category,total_sales
0,1001,Alex,Laptop,Computers,1200
1,1005,Maria,Laptop,Computers,1200
2,1002,Maria,Monitor,Accessories,600
3,1003,James,Keyboard,Accessories,300
4,1004,Alex,Monitor,Accessories,300


Category Summary

In [21]:
con.execute("""
    SELECT
        *
    FROM analytics.order_details
""").df()

,order_id,customer_name,state,product_name,category,quantity,price,total_sales
0,1001,Alex,CA,Laptop,Computers,1,1200,1200
1,1002,Maria,NV,Monitor,Accessories,2,300,600
2,1003,James,CA,Keyboard,Accessories,3,100,300
3,1004,Alex,CA,Monitor,Accessories,1,300,300
4,1005,Maria,NV,Laptop,Computers,1,1200,1200


In [24]:
con.execute("""
    SELECT
        category,
        COUNT(order_id) AS order_count,
        SUM(quantity) AS total_quantity,
        SUM(total_sales) AS total_revenue,
        AVG(total_sales) AS average_order_value
    FROM analytics.order_details
    GROUP BY category""").df()

,category,order_count,total_quantity,total_revenue,average_order_value
0,Accessories,3,6.0,1200.0,400.0
1,Computers,2,2.0,2400.0,1200.0


Hour 4 — Build the Warehouse Pipeline